In [6]:
import numpy as np
from pathlib import Path
import warnings

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)

DATA_DIR = Path("..") / "initial_data"

def suggest_ucb_point(
    X,
    y,
    beta=1.5,
    n_candidates=10_000,
    random_state=0,
    bias_point=None,
    bias_scale=0.1,
    bias_fraction=0.5
):
    """
    Given current data (X, y) for ONE function, suggest the next query point x* using
    a Gaussian Process + Upper Confidence Bound (UCB) heuristic.

    Parameters
    ----------
    X : np.ndarray, shape (N, d)
        The inputs already tried for this function.
        N = number of past points, d = input dimension (2, 3, 4, ..., 8 here).
    y : np.ndarray, shape (N,)
        The outputs observed for those inputs (same order as rows in X).
    beta : float
        Controls exploration vs exploitation in UCB = mean + beta * std.
        - Larger beta = more exploration (try uncertain points).
        - Smaller beta = more exploitation (stay near current best).
    n_candidates : int
        Base number of random candidate points to try in [0, 1]^d.
        We'll possibly increase this for higher dimensions.
    random_state : int
        Seed for the random number generator so results are reproducible.
    bias_point : np.ndarray or None
        If provided, sample some candidates near this point (for exploitation).
    bias_scale : float
        Standard deviation for biased sampling around bias_point.
    bias_fraction : float
        Fraction of candidates to sample near bias_point (rest are uniform).

    Returns
    -------
    best_x : np.ndarray, shape (d,)
        The chosen next input point (in [0, 1]^d) to submit as query.
    gp : GaussianProcessRegressor
        The fitted GP model, in case you want to inspect predictions later.
    """

    # --------------------------
    # 1. Basic info about X, y
    # --------------------------
    # X has shape (N, d): N = number of rows (past points), d = dimensions
    N, d = X.shape

    # --------------------------
    # 2. Build the GP kernel
    # --------------------------
    # RBF kernel:
    #   - Think of this as encoding the idea that nearby points in input space
    #     should have similar outputs (smoothness).
    #   - length_scale=np.ones(d) means we start by assuming each dimension
    #     has a length scale of 1.0; the GP can adapt this when fitting.
    #
    # WhiteKernel:
    #   - Adds some noise to the diagonal of the covariance matrix.
    #   - This allows the GP to handle noisy observations instead of trying
    #     to pass exactly through every point.
    kernel = 1.0 * RBF(length_scale=np.ones(d)) + WhiteKernel(noise_level=1e-3)

    # --------------------------
    # 3. Create and fit the GP
    # --------------------------
    gp = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,   # Centers/scales y internally, helpful when magnitudes differ
        random_state=random_state,
    )

    # Fit the GP on the current data (this is where it "learns" the function shape)
    gp.fit(X, y)

    # --------------------------
    # 4. Generate candidate points
    # --------------------------
    # We'll sample random points uniformly within [0, 1]^d.
    # For higher dimensions we increase the number of candidates because
    # the space is bigger and more "empty".
    if d <= 4:
        n = n_candidates
    else:
        # Double for d >= 5 to explore a bit more widely
        n = n_candidates * 2

    # modern NumPy RNG – local, reproducible
    rng = np.random.default_rng(random_state)

    if bias_point is not None:
        # Split candidates: some near bias_point, rest uniform
        n_biased = int(n * bias_fraction)
        n_uniform = n - n_biased
        
        # Biased candidates (normal distribution around bias_point)
        Xcand_biased = rng.normal(loc=bias_point, scale=bias_scale, size=(n_biased, d))
        Xcand_biased = np.clip(Xcand_biased, 0, 1)  # Keep in [0, 1]^d
        
        # Uniform candidates
        Xcand_uniform = rng.uniform(0.0, 1.0, size=(n_uniform, d))
        
        # Combine
        Xcand = np.vstack([Xcand_biased, Xcand_uniform])
    else:
        # Xcand has shape (n, d), with each coordinate between 0 and 1
        Xcand = rng.uniform(0.0, 1.0, size=(n, d))


    # --------------------------
    # 5. GP predictions at candidates
    # --------------------------
    # For each candidate point, we ask the GP for:
    #   - mu: predicted mean (what y value we expect)
    #   - std: predictive standard deviation (how uncertain we are)
    mu, std = gp.predict(Xcand, return_std=True)

    # --------------------------
    # 6. Compute UCB score
    # --------------------------
    # UCB(x) = mu(x) + beta * std(x)
    #   - mu(x): exploitation (high mean is good)
    #   - std(x): exploration (high uncertainty is good)
    # beta trades between them.
    ucb = mu + beta * std

    # I found that we had cases where the GP would suggest points that were
    # very close to points that were already evaluated. This was a problem
    # because the GP would then suggest the same point multiple times in a row.
    # This is a simple fix: avoid re-using any already-evaluated points.
    # Vectorized version: compute, in one NumPy broadcasted operation, which
    # candidate points are (within tolerance) equal to ANY seen point, instead
    # of looping over each seen point in Python. This removes Python-loop
    # overhead, scans the candidates once, and makes the intent ("mask any
    # candidate that matches any seen point") clearer.
    same_point = np.all(
        np.isclose(Xcand[:, None, :], X[None, :, :], atol=1e-8),
        axis=2,
    )  # shape: (n_candidates, N)
    mask_any_seen = np.any(same_point, axis=1)  # shape: (n_candidates,)
    ucb[mask_any_seen] = -np.inf

    # --------------------------
    # 7. Choose the best candidate
    # --------------------------
    # Take the candidate with the highest UCB value.
    best_idx = np.argmax(ucb)
    best_x = Xcand[best_idx]  # shape (d,)

    return best_x, gp


In [ ]:
# ============================
# Load multi-round inputs/outputs
# ============================

def parse_multiline_lists(filepath):
    """
    Parse a file where each list may span multiple lines.
    Lists start with '[array(' or '[np.' and end with ')]'.
    """
    with open(filepath) as f:
        content = f.read()
    
    # Join all lines and split on list boundaries
    # Each complete entry starts with '[' and ends with ')]'
    rounds = []
    buffer = ""
    bracket_count = 0
    
    for char in content:
        buffer += char
        if char == '[':
            bracket_count += 1
        elif char == ']':
            bracket_count -= 1
            if bracket_count == 0 and buffer.strip():
                rounds.append(buffer.strip())
                buffer = ""
    
    return rounds

input_lines = parse_multiline_lists("inputs_m13.txt")
output_lines = parse_multiline_lists("outputs_m13.txt")

inputs_rounds = [eval(line, {"np": np, "array": np.array}) for line in input_lines]
outputs_rounds = [eval(line, {"np": np}) for line in output_lines]

n_rounds = len(inputs_rounds)
assert len(outputs_rounds) == n_rounds, "Mismatch between input/output rounds"

# Sanity: each round must have 8 entries (functions 1–8)
for r in range(n_rounds):
    assert len(inputs_rounds[r]) == 8, f"Round {r} inputs != 8"
    assert len(outputs_rounds[r]) == 8, f"Round {r} outputs != 8"


beta_map = {
    1: 2.5,   # flat → explore MORE aggressively
    2: 1.0,   # strong → exploit near 0.611 region
    3: 2.0,   # still weak → explore
    4: 1.5,   # strong but dipped → more exploration around good region
    5: 0.5,   # unimodal, best known → exploit heavily with biased sampling
    6: 1.0,   # strong → exploit near -0.714 region
    7: 2.0,   # far from best → explore
    8: 1.0,   # strong → continue exploiting
}

for i in range(1, 9):
    # 1. Load initial data
    X_init = np.load(DATA_DIR / f"function_{i}" / "initial_inputs.npy")
    y_init = np.load(DATA_DIR / f"function_{i}" / "initial_outputs.npy")

    X = X_init.copy()
    y = y_init.copy()

    # print(y)

    # 2. Append every round's (x, y) for this function
    for r in range(n_rounds):
        x_prev = np.array(inputs_rounds[r][i - 1])
        y_prev = float(outputs_rounds[r][i - 1])

        print(f"Week {r + 1} {x_prev} -> {y_prev}")

        X = np.vstack([X, x_prev.reshape(1, -1)])
        y = np.append(y, y_prev)

    # print(y)

    # 3. Choose beta and call suggest_ucb_point as before
    beta = beta_map[i]

    if i == 5:
        # F5: biased sampling around best observed point
        best_idx = np.argmax(y)
        best_point = X[best_idx]

        best_x, gp = suggest_ucb_point(
            X,
            y,
            beta=beta, # e.g. 0.5 for heavy exploitation
            n_candidates=10_000,
            random_state=40 + i,
            bias_point=best_point,
            bias_scale=0.1,
            bias_fraction=0.7,
        )
    else:
        best_x, gp = suggest_ucb_point(
            X,
            y,
            beta=beta,
            n_candidates=10_000,
            random_state=40 + i,
        )


    # 6. Format new query for the portal
    query_str = "-".join(f"{v:.6f}" for v in best_x)

    # 7. Optional: show best y so far for context
    best_y_so_far = y.max()

    print(f"Function {i}:")
    print(f"  beta used: {beta}")
    print(f"  best y so far (after adding last week): {best_y_so_far:.6f}")
    print(f"  new query to submit: {query_str}")
    print()

Week 1 [0.954151 0.767932] -> 9.14184425020275e-88
Week 2 [0.125971 0.826988] -> -7.249963615484764e-176
Function 1:
  beta used: 2.5
  best y so far (after adding last week): 0.000000
  new query to submit: 0.849072-0.349783

Week 1 [0.773956 0.438878] -> 0.33661711576593556
Week 2 [0.858598 0.697368] -> 0.5919902522467393
Function 2:
  beta used: 1.0
  best y so far (after adding last week): 0.611205
  new query to submit: 0.094177-0.975622

Week 1 [0.652299 0.043775 0.02003 ] -> -0.14950216326847035
Week 2 [0.839213 0.587143 0.224705] -> -0.12367642933844279
Function 3:
  beta used: 2.0
  best y so far (after adding last week): -0.034835
  new query to submit: 0.751792-0.263692-0.419978

Week 1 [0.372882 0.447822 0.347304 0.45138 ] -> -0.25635124958376876
Week 2 [0.411939 0.431631 0.267425 0.398181] -> -1.2206425601891655
Function 4:
  beta used: 1.5
  best y so far (after adding last week): -0.256351
  new query to submit: 0.299820-0.414832-0.441612-0.439334

Week 1 [0.573131 0.528